In [1]:
from pynq import Overlay
import time

def write_128bit(bram, base_address, value_128):
    part0 = value_128 & 0xFFFFFFFF
    part1 = (value_128 >> 32) & 0xFFFFFFFF
    part2 = (value_128 >> 64) & 0xFFFFFFFF
    part3 = (value_128 >> 96) & 0xFFFFFFFF
    
    bram.write(base_address + 0x00, part0)
    bram.write(base_address + 0x04, part1)
    bram.write(base_address + 0x08, part2)
    bram.write(base_address + 0x0C, part3)

def read_128bit(bram, base_address):
    part0 = bram.read(base_address + 0x00)
    part1 = bram.read(base_address + 0x04)
    part2 = bram.read(base_address + 0x08)
    part3 = bram.read(base_address + 0x0C)
    
    # Costuramos tudo de volta arrastando os bits (<<) para a esquerda
    return (part3 << 96) | (part2 << 64) | (part1 << 32) | part0

def read_params(name, bram):
    with open(name, "r") as file:
        params = file.readlines() # Indentado corretamente
    
    offset = 0 # Cria o contador de endereços
    
    for linha_texto in params:
        linha_texto = linha_texto.strip() # Remove \n e espaços
        
        if not linha_texto:
            continue
            
        # Converte a string "0x..." para um número inteiro gigante
        valor = int(linha_texto, 16)
        
        # Grava na BRAM passando o BRAM, o endereço atual e o valor
        write_128bit(bram, offset, valor)
        
        # Pula 16 bytes (128 bits) para gravar a próxima linha no lugar certo
        offset += 16
        
    print(f"Gravação concluída: arquivo '{name}' carregado na BRAM!")
        
        
        
        
ol = Overlay("lenet5.bit")
bram_a = ol.bram_ctrl_a
bram_b = ol.bram_ctrl_b
bram_pesos = ol.bram_ctrl_pesos

gpio = ol.axi_gpio_0

gpio.channel1.write(0x00, 0xf)

read_params("conv1_params.txt", bram_pesos)
read_params("input_image.txt", bram_a)

Gravação concluída: arquivo 'conv1_params.txt' carregado na BRAM!
Gravação concluída: arquivo 'input_image.txt' carregado na BRAM!


In [2]:
print("Rodando camada C1...")
gpio.channel1.write(0x05, 0x1f) # 001 layer 1, reset 0, start 1 => 00101 = 5

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada C1 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0

Rodando camada C1...
Camada C1 Finalizou!


In [3]:
print(f"{read_128bit(bram_pesos, 12*16):032x}")

000000000000004d715c0c54e9a9420b


In [4]:
print("Rodando camada S2...")
gpio.channel1.write(0x09, 0x1f) # 010 layer 2, reset 0, start 1 => 01001 = 9

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada S2 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0

Rodando camada S2...
Camada S2 Finalizou!


In [5]:
read_params("conv2_params.txt", bram_pesos)

Gravação concluída: arquivo 'conv2_params.txt' carregado na BRAM!


In [6]:
print("Rodando camada C3...")
gpio.channel1.write(0x0D, 0x1f) # 011 layer 3, reset 0, start 1 => 01101 = 13

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada C3 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0

Rodando camada C3...
Camada C3 Finalizou!


In [7]:
print("Rodando camada S4...")
gpio.channel1.write(0x11, 0x1f) # 100 layer 2, reset 0, start 1 => 10001 = 17

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada S4 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0

Rodando camada S4...
Camada S4 Finalizou!


In [8]:
read_params("fc1_params.txt", bram_pesos)

Gravação concluída: arquivo 'fc1_params.txt' carregado na BRAM!


In [9]:
print("Rodando camada F6...")
gpio.channel1.write(0x15, 0x1f) # 101 layer 5, reset 0, start 1 => 10101 = 21

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada F6 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0 

Rodando camada F6...
Camada F6 Finalizou!


In [10]:
read_params("fc2_params.txt", bram_pesos)

Gravação concluída: arquivo 'fc2_params.txt' carregado na BRAM!


In [11]:
print("Rodando camada F6...")
gpio.channel1.write(0x19, 0x1f) # 110 layer 6, reset 0, start 1 => 11001 = 25

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada F6 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0 

Rodando camada F6...
Camada F6 Finalizou!


In [12]:
read_params("fc3_params.txt", bram_pesos)   

Gravação concluída: arquivo 'fc3_params.txt' carregado na BRAM!


In [13]:
print("Rodando camada F7...")
gpio.channel1.write(0x1D, 0x1f) # 111 layer 7, reset 0, start 1 => 11101 = 29

while (gpio.channel2.read() & 0x02) == 0:
    pass

print("Camada F7 Finalizou!")
gpio.channel1.write(0x00, 0x1f) # Start = 0 

Rodando camada F7...
Camada F7 Finalizou!


In [15]:
print(f"{read_128bit(bram_b, 0*16):032x}")

0000000000009c2d16dd22fe86ac5ac8
